# z304 – Entrenamiento Final + Submit
**Grupo 3: Banegas - Marín - Mengoni - Rey**

Entrena con el ensemble de semillas usando los mejores hiperparámetros de z303.
Mide WAPE in-sample. Genera predicciones para 202002. Submit a Kaggle.

| Input | Archivo |
|-------|---------|
| Features | `dataset_fe.parquet` (de z302) |
| Hiperparámetros | `z303_hiper_{modo}.json` (de z303) |

In [ ]:
import yaml, json, time, subprocess
from pathlib import Path
import numpy as np
import duckdb
import lightgbm as lgb
import mlflow
import pandas as pd
from google.cloud import storage as gcs

with open('../pipe_py/config.yaml') as f:
    CFG = yaml.safe_load(f)

# --- palancas ---
CFG['train']['semillas_ensemble'] = [102191, 42, 7, 13, 99]
SUBMIT = True   # False para solo generar CSV sin enviar
# ----------------

pr   = CFG['preproc']
modo = f"{pr['group_mode']}_{pr['missing_strategy']}_{pr['densify_strategy']}"
print(f'Modo: {modo}')

In [ ]:
# Cargar hiperparámetros
ruta_json = Path(CFG['paths']['optuna_out']) / f'z303_hiper_{modo}.json'
with open(ruta_json) as f:
    hiper = json.load(f)

best_params  = hiper['best_params']
tipo_target  = hiper['tipo_target']
feature_cols = hiper['features']
print(f'WAPE Optuna: {hiper["best_wape"]:.4f}  |  tipo_target: {tipo_target}')
print(f'Features: {len(feature_cols)}')

In [ ]:
# Función WAPE
def wape(y_true, y_pred):
    d = np.abs(y_true).sum()
    return np.abs(y_true - y_pred).sum() / d if d > 0 else 0.0

# Cargar datos de entrenamiento
ruta_fe = Path(CFG['paths']['fe_out']) / 'dataset_fe.parquet'
con = duckdb.connect()
con.execute(f"CREATE TABLE ds AS SELECT * FROM read_parquet('{ruta_fe}')")

cols_sql = ', '.join(feature_cols + ['target', 'B0', 'B1'])
datos = con.execute(f'SELECT {cols_sql} FROM ds').fetchnumpy()
X_tr = np.column_stack([datos[c] for c in feature_cols])
y_tr = datos['target']
B0   = datos.get('B0')
B1   = datos.get('B1')
print(f'X_train: {X_tr.shape}')

In [ ]:
# Entrenamiento ensemble
semillas = CFG['train']['semillas_ensemble']
modelos  = []
t0 = time.time()

for semilla in semillas:
    params = {**best_params, 'random_state': semilla, 'n_jobs': -1,
              'verbosity': -1, 'objective': CFG['optuna']['objective_lgbm']}
    m = lgb.LGBMRegressor(**params)
    m.fit(X_tr, y_tr, callbacks=[lgb.log_evaluation(-1)])
    modelos.append(m)
    print(f'  semilla {semilla} → OK')

print(f'Ensemble entrenado en {time.time()-t0:.0f}s')

In [ ]:
# WAPE in-sample (diagnóstico)
pred_tr = np.stack([m.predict(X_tr) for m in modelos]).mean(axis=0)

if tipo_target == 'delta' and B0 is not None:
    pred_nivel = pred_tr * B1 + B0
    real_nivel = y_tr * B1 + B0
else:
    pred_nivel = pred_tr
    real_nivel = y_tr

wape_is = wape(real_nivel, pred_nivel)
print(f'WAPE in-sample (nivel): {wape_is:.4f}')

In [ ]:
# Feature importance (promedio del ensemble)
importances = np.stack([m.feature_importances_ for m in modelos]).mean(axis=0)
fi_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(20)
fi_df.set_index('feature').plot(kind='barh', figsize=(8,6), title='Top 20 features')

In [ ]:
# Inferencia 202002
# NOTA: dataset_fe debe incluir fila de período de inferencia
periodo_inf = CFG['train']['periodo_inferencia']
has_periodo = 'periodo' in [r[0] for r in con.execute('DESCRIBE ds').fetchall()]

if has_periodo:
    datos_inf = con.execute(
        f"SELECT {', '.join(feature_cols + ['B0','B1'])} FROM ds WHERE periodo = {periodo_inf}"
    ).fetchnumpy()
else:
    datos_inf = datos  # fallback: mismos datos

X_inf  = np.column_stack([datos_inf[c] for c in feature_cols])
B0_inf = datos_inf.get('B0')
B1_inf = datos_inf.get('B1')

pred_inf = np.stack([m.predict(X_inf) for m in modelos]).mean(axis=0)
if tipo_target == 'delta' and B0_inf is not None:
    pred_nivel_inf = pred_inf * B1_inf + B0_inf
else:
    pred_nivel_inf = pred_inf

pred_nivel_inf = np.clip(pred_nivel_inf, 0, None)
print(f'Predicciones: {len(pred_nivel_inf):,} productos | min={pred_nivel_inf.min():.2f} max={pred_nivel_inf.max():.2f}')

In [ ]:
# Guardar submit CSV
ruta_submit = Path(CFG['paths']['submit_out']) / f'submit_{modo}.csv'
ruta_submit.parent.mkdir(parents=True, exist_ok=True)
df_submit = pd.DataFrame({'product_id': range(len(pred_nivel_inf)), 'tn': pred_nivel_inf})
df_submit.to_csv(ruta_submit, index=False)
print(f'Submit guardado → {ruta_submit}')
df_submit.describe()

In [ ]:
# Subir a GCS
g = CFG['gcs']
client = gcs.Client()
bucket = client.bucket(g['bucket'])
blob = bucket.blob(f"{g['prefix_submit']}/submit_{modo}.csv")
blob.upload_from_filename(str(ruta_submit))
print(f"Subido → gs://{g['bucket']}/{g['prefix_submit']}/submit_{modo}.csv")

In [ ]:
# MLflow log
mlflow.set_tracking_uri(CFG['paths']['mlflow_uri'])
mlflow.set_experiment(CFG['optuna']['experiment_name'])
with mlflow.start_run(run_name=f'train_{modo}'):
    mlflow.log_metric('wape_insample', wape_is)
    mlflow.log_params({'modo': modo, 'tipo_target': tipo_target,
                       'n_semillas': len(semillas)})
    mlflow.log_artifact(str(ruta_submit))
print('Logueado en MLflow')

In [ ]:
# Kaggle submit (opcional)
if SUBMIT:
    competencia = CFG['train']['kaggle_competition']
    mensaje = f'labo3 grupo3 modo={modo} wape_is={wape_is:.4f}'
    cmd = ['kaggle', 'competitions', 'submit', '-c', competencia,
           '-f', str(ruta_submit), '-m', mensaje]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f'Error: {result.stderr}')
else:
    print('Submit omitido (SUBMIT=False)')